In [ ]:
import itertools
import logging
import pathlib

import altair as alt
import geopandas as gpd
import numpy as np
import pandas as pd
import shapely
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
from scipy.optimize import minimize
from scipy.stats import norm

import pitchmark.osm

alt.renderers.enable('mimetype')
logger = logging.getLogger(__name__)

In [ ]:
METER_PER_FOOT = 0.305
METER_PER_YARD = 3 * METER_PER_FOOT
METER_PER_INCH = METER_PER_FOOT / 12
GRAVITY = 9.8 / METER_PER_YARD  # yd/s^2
MOI_SOLID_SPHERE = 0.4
BALL_RADIUS = 0.84 / 36  # yd
HOLE_RADIUS = 2.125 / 36  # yd

## Real green (Augusta National 16th)

In [ ]:
map_path = pathlib.Path().cwd().parent / "data-raw" / "OpenStreetMap" / "augusta_national" / "map.osm"
handler = pitchmark.osm.GolfHandler()
handler.apply_file(map_path)
fc = handler.feature_collection
augusta_national = pitchmark.Course.from_featurecollection(fc)

In [ ]:
redbud = augusta_national.holes[15]
redbud_flag = augusta_national.transformer_to_local.transform(*redbud.path.coords[-1])
redbud_green = redbud.gdf[
    redbud.gdf.contains(shapely.Point(redbud_flag))
    & (redbud.gdf["course_area"] == "putting_green")
].unary_union
redbud_green_surround = redbud_green.buffer(5.0)
shapely.prepare(redbud_green_surround)
redbud_green_surround

In [ ]:
redbud_mesh_file = pathlib.Path().cwd().parent / "data-raw" / "USGS_LIDAR" / "redbud_mesh.csv"
redbud_mesh_df = pd.read_csv(redbud_mesh_file, index_col=0)
redbud_mesh_df["geometry"] = gpd.GeoSeries.from_wkt(redbud_mesh_df["geometry"])
redbud_mesh_df

In [ ]:
redbud_gdf = gpd.GeoDataFrame(redbud_mesh_df, crs=redbud.gdf.crs)
redbud_green_gdf = redbud_gdf[redbud_gdf.within(redbud_green)]
redbud_green_gdf

In [ ]:
surround_features = pitchmark.plotting.chart_course(redbud.gdf.clip(redbud_green_surround))

In [ ]:
grades = (alt.Chart(redbud_green_gdf)
    .mark_geoshape(
        filled=True,
        color="lightgray",
    )
    .encode(
        color=alt.Color("slope_grade:Q", scale=alt.Scale(domain=[0, 12.0], scheme="greys"))
    )
    .project(
        type="identity",
        reflectY=True,
    )
)

In [ ]:
hover = alt.selection_single(
    name='hover',
    on='mouseover',
    nearest=True,
    empty='none'
)
zoom = alt.selection_interval(
    name='zoom',
    bind='scales'
)
inclines = (
    alt.Chart(redbud_green_gdf)
    .mark_point(
        shape="wedge",
        filled=True,
    )
    .encode(
        longitude="x",
        latitude="y",
        color=alt.condition(
            hover,
            alt.value("red"),
            alt.value("black")
        ),
        angle="slope_heading",
        size="slope_grade",
        tooltip=["x", "y", "z", "slope_heading", "slope_grade"],
    )
    .add_selection(hover)
    .project(
        type="identity",
        reflectY=True,
    )
    .properties(
        width=800,
        height=800,
    )
)


In [ ]:
hole_location = (-681.0, -339.5)
hole_disk = gpd.GeoDataFrame(
    geometry=[shapely.buffer(shapely.Point(hole_location), HOLE_RADIUS)],
)
hole_disk_exaggerated = gpd.GeoDataFrame(
    geometry=[shapely.buffer(shapely.Point(hole_location), 5*HOLE_RADIUS)],
)
hole_plot = (
    alt.Chart(hole_disk)
    .mark_geoshape(color="black")
)
hole_plot_exaggerated = (
    alt.Chart(hole_disk_exaggerated)
    .mark_geoshape(color="black")
)
hole_plot

In [ ]:
surround_features + grades + inclines + hole_plot_exaggerated

In [ ]:
def softness(stimp_reading, *, ball_moi = MOI_SOLID_SPHERE):
    init_ball_speed = 2  # yd/s, stimp_reading is in ft = yd/3
    return (init_ball_speed**2) * (1.0 + ball_moi) / (2 * GRAVITY * stimp_reading / 3)

When repeatedly accessing a GeoDataFrame using the geometry, a STRtree is efficient.
This needs to be constructed manually so as to be reusable in many accessions.

In [ ]:
softness

In [ ]:
2.0*2.0 * 1.4 / (2.0 * 10.7 * 12.0/3)

In [ ]:
class Green:
    def __init__(
        self,
        stimp,
        *,
        gdf=None,
        hole_location=None,
        hole_radius=HOLE_RADIUS,
        holing_vmax=1.63
    ):
        self.stimp = stimp
        self.softness = softness(stimp, ball_moi=MOI_SOLID_SPHERE)
        self.gdf = gdf
        if gdf is not None:
            self.strtree = shapely.STRtree(gdf.geometry)
        self.hole_location = (0.0, 0.0) if hole_location is None else hole_location
        self.hole_radius = hole_radius
        self.hole_radius_squared = hole_radius * hole_radius
        self.holing_vmax = holing_vmax
        
    def normal(self, x, y):
        point = shapely.Point(x, y)
        candidate_indices = self.strtree.query(point, predicate="within")
        try:
            gdf_index = candidate_indices[0]  # expects single hit
            row = self.gdf.iloc[gdf_index]
            return row[["normal_x", "normal_y", "normal_z"]].tolist()
        except IndexError:
            print(f"No candidate polygon found for {point=}\n")
            return([0.0, 0.0, 1.0])
    
    def squared_distance_to_hole(self, x, y):
        x0, y0 = self.hole_location
        return (x - x0) * (x - x0) + (y - y0) * (y - y0)
    
    def distance_to_hole(self, *args):
        return np.sqrt(self.squared_distance_to_hole(*args))

    def impact_function(self, u):
        x, y, vx, vy = u
        impact = self.squared_distance_to_hole(x, y) / self.hole_radius_squared
        v = np.sqrt(vx * vx + vy * vy)
        return v - self.holing_vmax * (1.0 - impact)
    


In [ ]:
redbud_green_obj = Green(
    12.0,
    gdf=redbud_green_gdf,
    hole_location=hole_location,
)

In [ ]:
redbud_green_obj.strtree.query(shapely.Point(-670.0, -340.0), predicate="within")[0]

In [ ]:
row = redbud_green_gdf.iloc[1239]
row[["x", "y", "z"]].tolist()

Recall the Stimpmeter result

$$
y = - \frac{v_0^2}{2 a_y}
$$

and

$$
a_y = \frac{- \rho_g g}{1 + \frac{I}{m R^2}} = -\frac{5}{7} \rho_g g
$$

such that, to travel a distance $y$, the initial velocity should be

$$
v_0 = \sqrt{2 a_y y} = \sqrt{\frac{10}{7} \rho_g g y}
$$

Everything cancels with the softness calculation to be in terms of the stimp reading:

$$
v_{init} = (2.0 \textrm{yd/s}) \frac{\delta x}{X_s}
$$

with $X_s$ the stimp reading in yards.

In [ ]:
redbud_green_obj.softness

In [ ]:
init_ball_position = (-682.0, -337.8)
hole_location = redbud_green_obj.hole_location
dx = hole_location[0] - init_ball_position[0]
dy = hole_location[1] - init_ball_position[1]
du = np.sqrt(dx*dx + dy*dy)
theta_init = np.arctan2(dy, dx)
v_init = 2.0 * np.sqrt(du / (redbud_green_obj.stimp / 3))
print(du, v_init, theta_init)

In [ ]:
du

In [ ]:
2 * du * redbud_green_obj.softness * GRAVITY / (1.0 + MOI_SOLID_SPHERE)

In [ ]:
redbud_green_obj.distance_to_hole(*init_ball_position)

In [ ]:
print(dy, dx)

In [ ]:
def simple_roll(t, u, green):
    x, y, vx, vy = u
    dx = vx
    dy = vy

    g = GRAVITY
    I_b = MOI_SOLID_SPHERE
    rho_g = green.softness

    nx, ny, nz = green.normal(x, y)
    v = np.sqrt(vx * vx + vy * vy)
    cos_theta = vx / v
    sin_theta = vy / v

    prefactor = - g * (I_b) / (1.0 + I_b)

    dv_forward = prefactor * (rho_g / I_b + nx * cos_theta - ny * sin_theta)
    dv_perpendicular = prefactor * (nx * sin_theta + ny * cos_theta)

    dvx = dv_forward * cos_theta - dv_perpendicular * sin_theta
    dvy = dv_forward * sin_theta + dv_perpendicular * cos_theta
    return [dx, dy, dvx, dvy]

In [ ]:
point_init = shapely.Point(init_ball_position[0], init_ball_position[1])
candidate_indices = redbud_green_obj.strtree.query(point_init, predicate="within")
gdf_index = candidate_indices[0]  # expects single hit
row = redbud_green_obj.gdf.iloc[gdf_index]
row[["normal_x", "normal_y", "normal_z"]].tolist()

In [ ]:
class MinBallSpeed:
    def __init__(self, min_speed, *, terminal=True, direction=0):
        self.terminal = terminal
        self.direction = direction
        self.min_speed = min_speed

    def __call__(self, t, u, green):
        x, y, vx, vy = u
        v2 = vx * vx + vy * vy
        return v2 - self.min_speed * self.min_speed

In [ ]:
def holing_objective(sol, green):
    i_nearest = np.argmin([green.impact_function(u) for u in sol.y.T])
    try:
        t_min = sol.t[i_nearest - 1]
    except IndexError:
        t_min = sol.t[0]
    try:
        t_max = sol.t[i_nearest + 1]
    except IndexError:
        t_max = sol.t[-1]
    t_eval = np.arange(t_min, t_max + 0.01, 0.01)
    u_eval = sol.sol(t_eval).T
    impact = np.amin([green.impact_function(u) for u in u_eval])
    return impact

In [ ]:
def simulate_rolls(vs, thetas, green, init, *, tspan=(0.0, 10.0), method="LSODA", min_ball_speed_tol=1e-3, **kwargs):
    x0, y0 = init
    end_points = list()
    sols = list()
    impacts = list()
    for v, theta in itertools.product(vs, thetas):
        vx0, vy0 = v * np.cos(theta), v * np.sin(theta)
        u0 = np.array([x0, y0, vx0, vy0])
        sol = solve_ivp(
            simple_roll,
            tspan,
            u0,
            method=method,
            dense_output=True,
            events=[MinBallSpeed(min_ball_speed_tol)],
            args=(green,),
            **kwargs,
        )
        end_point = np.hstack(([v, theta], sol.y.T[-1]))
        impact = holing_objective(sol, green)
        end_points.append(end_point)
        sols.append(sol)
        impacts.append(impact)

    return end_points, sols, impacts

In [ ]:
end_points, sols, impacts = simulate_rolls([v_init], [theta_init], redbud_green_obj, init_ball_position)

In [ ]:
sol = sols[0]

In [ ]:
sol_df = pd.DataFrame(np.vstack(([sol.t], sol.y)).T, columns=["t", "x", "y", "vx", "vy"])
sol_df["v"] = np.sqrt(sol_df["vx"]**2 + sol_df["vy"]**2)
sol_df

In [ ]:
sol_points = alt.Chart(sol_df).mark_point(clip=True).encode(
    longitude="x",
    latitude="y",
    color="v",
)
hole_plot + sol_points

In [ ]:
surround_features + grades + inclines + hole_plot_exaggerated + sol_points

In [ ]:
broadie_putts_gained = pd.DataFrame(
    {
        "distance_ft": [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 30, 40, 50, 60, 90],
        "avg_putts": [
            1.0,
            1.01,
            1.05,
            1.14,
            1.24,
            1.34,
            1.43,
            1.5,
            1.56,
            1.61,
            1.78,
            1.87,
            1.98,
            2.06,
            2.14,
            2.21,
            2.36,
        ],
    }
)
approx_putts = interp1d(
    broadie_putts_gained["distance_ft"] / 3,  # yd
    broadie_putts_gained["avg_putts"],
    fill_value="extrapolate"
)


In [ ]:
approx_putts(6.5 * 3)

In [ ]:
def aim_objective_inner(x, green, init_position, *, v_cv=0.06, theta_sd=np.deg2rad(1.0)):
    v, theta = x
    logging.debug(f"Simulating rolls for bundle centered at {v=}, {theta=}...")
    vs = v * np.geomspace(1.0 - 2 * v_cv, 1.0 + 2 * v_cv, 9)
    thetas = theta + np.linspace(-2 * theta_sd, 2 * theta_sd, 9)
    return simulate_rolls(vs, thetas, green, init_position)


In [ ]:
square_block_2sigma = itertools.product(np.linspace(-2.0, 2.0, 9), np.linspace(-2.0, 2.0, 9))

weights = [
    norm.pdf(x) * norm.pdf(y)
    for x, y in square_block_2sigma
]
weights /= sum(weights)


def approximate_scores(end_points, impacts, green, *, putting_approximant=approx_putts):
    return [
        putting_approximant(green.distance_to_hole(u[2], u[3])) if impact >= 0 else 0
        for u, impact in zip(end_points, impacts)
    ]

def aim_objective(x, green, init_position, **kwargs):
    try:
        end_points, _, impacts = aim_objective_inner(x, green, init_position, **kwargs)
    except:
        return np.Inf
    scores = approximate_scores(end_points, impacts, green)

    return sum(score * weight for score, weight in zip(scores, weights))

In [ ]:
end_points, sols, impacts = aim_objective_inner((v_init, theta_init), redbud_green_obj, init_ball_position)

In [ ]:
df = pd.DataFrame(end_points, columns=["v_init", "theta", "x", "y", "vx_end", "vy_end"])
df["impact"] = impacts
df["holed"] = df["impact"] < 0.0
df["score"] = approximate_scores(end_points, impacts, redbud_green_obj)

In [ ]:
end_points_cloud = (
    alt.Chart(df)
    .mark_point(filled=True)
    .encode(
        longitude="x",
        latitude="y",
        color=alt.condition(
            hover,
            alt.value("red"),
            "score:Q",
        ),
        shape="holed",
        tooltip=["v_init", "theta", "x", "y", "impact", "score:Q"],
    )
    # .project(
    #     type="identity",
    #     reflectY=True,
    # )
    .add_selection(hover)
)
hole_plot + end_points_cloud

In [ ]:
surround_features + grades + hole_plot_exaggerated + end_points_cloud

In [ ]:
def take_trajectory(sol):
    df = pd.DataFrame(sol.y.T, columns=["x", "y", "vx", "vy"])
    df["t"] = sol.t
    return(df)

middle_trajectory = take_trajectory(sols[len(sols) // 2])
middle_trajectory_plot = (
    alt.Chart(middle_trajectory)
    .mark_line(color="black", strokeWidth=0.5)
    .encode(
        longitude="x",
        latitude="y",
        detail="sol_index:N"
    )
    .properties(width=600, height=400)
)
middle_trajectory_plot

In [ ]:
surround_features + grades + hole_plot_exaggerated + end_points_cloud + middle_trajectory_plot

In [ ]:
logger.setLevel(logging.DEBUG)
optim_result = minimize(
    aim_objective,
    (v_init, theta_init),
    args=(redbud_green_obj, init_ball_position),
    method="Nelder-Mead",
    bounds=[(0.0, 20.0), (-np.pi, np.pi)],
    options={"maxiter": 10, "disp": True},
)
optim_result

In [ ]:
end_points, sols, impacts = aim_objective_inner(optim_result.x, redbud_green_obj, init_ball_position)

In [ ]:
df = pd.DataFrame(end_points, columns=["v_init", "theta", "x", "y", "vx_end", "vy_end"])
df["impact"] = impacts
df["holed"] = df["impact"] < 0.0
df["score"] = approximate_scores(end_points, impacts, redbud_green_obj)

In [ ]:
end_points_cloud = (
    alt.Chart(df)
    .mark_point(filled=True)
    .encode(
        longitude="x",
        latitude="y",
        color=alt.condition(
            hover,
            alt.value("red"),
            "score:Q",
        ),
        shape="holed",
        tooltip=["v_init", "theta", "x", "y", "impact", "score:Q"],
    )
    # .project(
    #     type="identity",
    #     reflectY=True,
    # )
    .add_selection(hover)
    .properties(width=400, height=400)
)
hole_plot + end_points_cloud

In [ ]:
middle_trajectory = take_trajectory(sols[len(sols) // 2])
middle_trajectory_plot = (
    alt.Chart(middle_trajectory)
    .mark_line(color="black", strokeWidth=0.5)
    .encode(
        longitude="x",
        latitude="y",
        detail="sol_index:N"
    )
    .properties(width=600, height=400)
)

In [ ]:
surround_features + grades + hole_plot_exaggerated + end_points_cloud + middle_trajectory_plot